# yolov8_cpp — COCO128 training on a free Colab GPU (pure C++, CUDA build)

Verifies the **unified `yolo` CLI** (dataset ingestion + augmentation) built with `nvcc -DUSE_CUDA` and trained on **COCO128**. No Python at run time — the C++ binary does everything.

**First: Runtime → Change runtime type → GPU (T4).** Then Run all.


In [ ]:
!nvidia-smi -L
!nvcc --version | tail -1


### 1. Clone the repo (ships `yolov8n.pt` + arch manifest/names/depths)


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### 2. Get COCO128 (standard Ultralytics YOLO layout: images/ + labels/, normalised labels)


In [ ]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip
!unzip -q -o coco128.zip -d /content/yolov8_cpp
!ls coco128/images/train2017 | wc -l  # 128 images


### 3. Write a `data.yaml` (COCO128 uses its train split for val too)


In [ ]:
%%writefile coco128.yaml
path: /content/yolov8_cpp/coco128
train: images/train2017
val: images/train2017
nc: 80
names: [person, bicycle, car, motorcycle, airplane, bus, train, truck, boat, traffic_light, fire_hydrant, stop_sign, parking_meter, bench, bird, cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe, backpack, umbrella, handbag, tie, suitcase, frisbee, skis, snowboard, sports_ball, kite, baseball_bat, baseball_glove, skateboard, surfboard, tennis_racket, bottle, wine_glass, cup, fork, knife, spoon, bowl, banana, apple, sandwich, orange, broccoli, carrot, hot_dog, pizza, donut, cake, chair, couch, potted_plant, bed, dining_table, toilet, tv, laptop, mouse, remote, keyboard, cell_phone, microwave, oven, toaster, sink, refrigerator, book, clock, vase, scissors, teddy_bear, hair_drier, toothbrush]


### 4. Build `make_init_pt` (host g++) and make the initial `.pt` from the shipped `yolov8n.pt` — pure C++, no Python


In [ ]:
!g++ -std=c++20 -O2 -Ipure/third_party pure/make_init_pt.cpp -o make_init_pt
!./make_init_pt init.pt from yolov8n.pt


### 5. Build the unified `yolo` CLI as **CUDA** (`nvcc -DUSE_CUDA`)
First nvcc build of the new CLI — if it errors, copy the tail back so we can fix it.


In [ ]:
!nvcc -x cu -std=c++17 --extended-lambda -DUSE_CUDA -O2 -Ipure/third_party pure/yolo.cpp -o yolo 2>&1 | tail -40
!ls -la yolo


### 6. Sanity run — 3 epochs, 640px, **fine-tune LR** (confirms it trains on GPU + timing)
Baseline: the pretrained `init.pt` already vals ~0.59 mAP@0.5 on COCO128, so fine-tuning
should stay near/above that — NOT collapse. `--lr` defaults to 1e-3 (2e-3 destroys pretrained).


In [ ]:
%%time
!./yolo train --data coco128.yaml --weights init.pt --imgsz 640 --epochs 3 --batch 4 --lr 2e-4


### 7. Full run — 100 epochs (optional; long). Adjust epochs/batch to taste.


In [ ]:
%%time
!./yolo train --data coco128.yaml --weights init.pt --imgsz 640 --epochs 100 --batch 8 --lr 1e-3 --mosaic 1 --mixup 1 --close-mosaic 10


### 8. Validate (`best.pt`) — reports mAP@0.5 and mAP@0.5:0.95


In [ ]:
!./yolo val --data coco128.yaml --weights best.pt --imgsz 640


### 9. Detect on one image + show it


In [ ]:
!./yolo detect --weights best.pt --source coco128/images/train2017/000000000009.jpg --imgsz 640 --conf 0.25 --out det.png --data coco128.yaml
from IPython.display import Image; Image('det.png')


---
**Notes / caveats**
- This is the first `nvcc -DUSE_CUDA` build of the new dataset/aug CLI. The aug + dataset code is host-side; only conv/matmul route to the GPU via the `bk::` seam.
- The GEMM backend is *hosted* (copies host↔device per op), so the GPU speedup may be limited until device-resident buffers land (a RESUME item). Still far faster than CPU.
- Estimate: COCO128/640px/100ep on a T4 ≈ tens of minutes (CPU would be ~a day).
- `best.pt` is a standard state_dict — `torch.load` / Ultralytics `load_state_dict` can read it.
